In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import pandas as pd

from dotenv import dotenv_values

In [3]:
# Load config file agnostic to project or notebook work directory
config_path = Path("config.env")
if not config_path.exists():
    config_path = Path("../../../config.env")
CONFIG = dotenv_values(config_path)
assert CONFIG != {}, 'Configuration is empty - check the filepath.'

# Main data directory
DATADIR = Path(CONFIG['DATADIR'])

# Submodule specific data directory
OUTDIR = DATADIR / '10_analyses' / 'sexual_dimorphism'
OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# Identify all multivariate result files via a glob search of result directory
fp_mv_results_dir = OUTDIR / 'L4_multivariate_sex_results'
fp_mv_results = list(fp_mv_results_dir.glob('*4-*.txt'))
print(f"{len(fp_mv_results)} multivaraite result files were found!")
mv_result_ids = [path.stem.split()[0] for path in fp_mv_results]

708 multivaraite result files were found!


In [5]:
# Load all files into Pandas DataFrames
dict_mv_results = {}
for id_, path in zip(mv_result_ids, fp_mv_results):
    df = pd.read_csv(path, delimiter='\t', index_col=0)
    df['id'] = id_
    
    dict_mv_results[id_] = df

# Concatenate into a full DataFrame
df_mv_results = pd.concat(dict_mv_results.values())

In [6]:
# Save out to OUTDIR
outfile = OUTDIR / 'L4_multivariate_sex_edgeR_results_full.csv'
df_mv_results.to_csv(outfile)

In [8]:
df_sig_mv_results = df_mv_results[df_mv_results['FDR'] < 0.05]

outfile = OUTDIR / 'L4_multivariate_sex_edgeR_results_significant_only.csv'
df_sig_mv_results.to_csv(outfile)